# VANTA MobileNetV3-Small training on Kaggle

Attach two Kaggle inputs before running: (1) the VANTA repository or `training/` directory and (2) the unchanged AlphaTrash checkout containing `trash_dataset/`. Enable a GPU accelerator. If notebook internet is disabled, also attach the official TorchVision MobileNetV3-Small ImageNet weight file and set `PRETRAINED_WEIGHTS` below. The full training cell is disabled by default.

In [ ]:
from pathlib import Path
import json
import shutil
import subprocess
import sys

# Change only these input paths to match the names shown in Kaggle's Input panel.
PROJECT_INPUT = Path('/kaggle/input/vanta-source/VANTA/training')
DATASET_ROOT = Path('/kaggle/input/alphatrash-dataset/trash_dataset')
PRETRAINED_WEIGHTS = None  # Example: Path('/kaggle/input/mobilenet-v3-small-weights/mobilenet_v3_small-047dcff4.pth')

EXPECTED_MANIFEST_ID = '5fe461138fdbc07c2b591209d9865a50db14420fca9f34e2ec45e55914586464'
SOURCE_REVISION = '33801d876c161b1e35073ebd401a6e2a0cdece3b'
WORKING_CODE = Path('/kaggle/working/vanta-training-code')
OUTPUT_DIR = Path('/kaggle/working/vanta-output')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

if not PROJECT_INPUT.is_dir():
    raise FileNotFoundError(f'Adjust PROJECT_INPUT; directory not found: {PROJECT_INPUT}')
if not DATASET_ROOT.is_dir():
    raise FileNotFoundError(f'Adjust DATASET_ROOT; directory not found: {DATASET_ROOT}')
if PRETRAINED_WEIGHTS is not None and not PRETRAINED_WEIGHTS.is_file():
    raise FileNotFoundError(f'Pretrained weights not found: {PRETRAINED_WEIGHTS}')

# Kaggle inputs are read-only, so copy only the small training code—not the images.
shutil.copytree(PROJECT_INPUT, WORKING_CODE, dirs_exist_ok=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '--no-deps', '--no-build-isolation', '-e', str(WORKING_CODE)], check=True)
print(f'Code: {WORKING_CODE}')
print(f'Dataset: {DATASET_ROOT}')
print(f'Outputs: {OUTPUT_DIR}')

## Reproduce and verify the clean manifest

This scans the attached AlphaTrash input, regenerates the split manifest, and stops if its ID differs from the locally verified ID.

In [ ]:
generated_manifest = OUTPUT_DIR / 'alphatrash_clean_splits.json'
conflict_report = OUTPUT_DIR / 'duplicate_label_conflicts.json'
subprocess.run([
    sys.executable, str(WORKING_CODE / 'scripts/create_clean_manifest.py'),
    '--data-dir', str(DATASET_ROOT),
    '--output', str(generated_manifest),
    '--conflict-report', str(conflict_report),
    '--source-revision', SOURCE_REVISION,
], check=True)
manifest = json.loads(generated_manifest.read_text())
if manifest['manifest_id'] != EXPECTED_MANIFEST_ID:
    raise RuntimeError(f"Manifest mismatch: expected {EXPECTED_MANIFEST_ID}, got {manifest['manifest_id']}")
print(f"Verified manifest ID: {manifest['manifest_id']}")

## CUDA smoke test

The smoke test validates every manifest reference and hash, loads ImageNet weights once, loads train/validation batches with two workers, and performs one mixed-precision optimizer step plus one validation forward pass. It does not use the test split for inference and does not save a model.

In [ ]:
smoke_command = [
    sys.executable, str(WORKING_CODE / 'scripts/kaggle_smoke_test.py'),
    '--data-dir', str(DATASET_ROOT),
    '--manifest', str(generated_manifest),
    '--expected-manifest-id', EXPECTED_MANIFEST_ID,
    '--output', str(OUTPUT_DIR / 'smoke_test.json'),
    '--batch-size', '16', '--workers', '2', '--seed', '42',
]
if PRETRAINED_WEIGHTS is not None:
    smoke_command.extend(['--pretrained-weights', str(PRETRAINED_WEIGHTS)])
subprocess.run(smoke_command, check=True)

## Full training (disabled by default)

Inspect the smoke-test output first. Set `RUN_FULL_TRAINING = True` only when ready. This performs model selection with train/validation data only; it does not evaluate the final test split.

In [ ]:
RUN_FULL_TRAINING = False
training_command = [
    sys.executable, str(WORKING_CODE / 'scripts/train.py'),
    '--data-dir', str(DATASET_ROOT),
    '--manifest', str(generated_manifest),
    '--checkpoint', str(OUTPUT_DIR / 'best_model.pt'),
    '--metrics', str(OUTPUT_DIR / 'training_metrics.json'),
    '--class-mapping', str(OUTPUT_DIR / 'class_mapping.json'),
    '--validation-report', str(OUTPUT_DIR / 'cleaned_dataset_validation.json'),
    '--device', 'cuda', '--batch-size', '16', '--workers', '2',
    '--stage-a-epochs', '10', '--stage-a-learning-rate', '0.001',
    '--stage-b-epochs', '8', '--stage-b-learning-rate', '0.0001',
    '--unfreeze-blocks', '3', '--early-stopping-patience', '3', '--seed', '42',
]
if PRETRAINED_WEIGHTS is not None:
    training_command.extend(['--pretrained-weights', str(PRETRAINED_WEIGHTS)])
print(' '.join(training_command))
if RUN_FULL_TRAINING:
    subprocess.run(training_command, check=True)
else:
    print('Full training was not started. Set RUN_FULL_TRAINING=True when ready.')

## Package downloadable outputs

Run after training. Kaggle preserves files under `/kaggle/working` when you save a notebook version.

In [ ]:
required_outputs = [
    'best_model.pt', 'training_metrics.json', 'class_mapping.json',
    'cleaned_dataset_validation.json', 'smoke_test.json',
    'alphatrash_clean_splits.json', 'duplicate_label_conflicts.json',
]
missing_outputs = [name for name in required_outputs if not (OUTPUT_DIR / name).is_file()]
if missing_outputs:
    raise FileNotFoundError(f'Training outputs are incomplete: {missing_outputs}')
archive_path = shutil.make_archive('/kaggle/working/vanta-training-artifacts', 'zip', OUTPUT_DIR)
print(f'Download: {archive_path}')